# Examen Primer Parcial — Laboratorio de Análisis de Datos Financieros y Diseño de Indicadores

**Duración:** 1 hora 40 minutos
**Módulos cubiertos:**

* **Módulo 1:** Herramientas de análisis de datos y definición de métricas financieras clave.
* **Módulo 2:** Evaluación de la madurez analítica y elaboración de reportes ejecutivos.

---


## Instrucciones Generales

* Este examen usa el **método del caso** con dos tablas: **Usuarios** y **Compras**.
* Muestra todos tus **cálculos** y **explica tus decisiones** de manera breve y directa.
* Usa **Python (Pandas, Matplotlib, Seaborn)** en Jupyter.
* Coloca tus respuestas **debajo de cada celda** indicada.
* Al final, redacta una **conclusión ejecutiva** (3–6 frases) para dirección.
* **Entrega:** Notebook ejecutado y limpio (sin errores), con los gráficos realizados.

---

# Caso: **Shoply — Marketplace de Productos Digitales**

**Contexto**
Shoply vende productos en línea a minoristas y mayoristas. El equipo sospecha problemas de **retención** y **monetización** y requiere un análisis para entender patrones de compra, **Churn**, **ARPU** y **LTV**.

**Tu rol**
Eres analista financiero. Debes **limpiar datos**, calcular **KPIs**, explorar **relaciones** con variables y proponer **acciones**.

---



## Archivos de trabajo (debes pegarlos como CSV)

1. **usuarios.csv**
   | Columna | Descripción |
   |---|---|
   | `user_id` | ID único del usuario |
   | `age` | Edad |
   | `gender` | Sexo ("M"/"F") |
   | `country` | País |
   | `signup_channel` | Canal (Ads, SEO, Referral, Direct) |
   | `monthly_income` | Ingreso mensual estimado (USD) |
   | `subscription_type` | Plan (Free, Basic, Premium, Pro) |
   | `support_tickets` | # de tickets de soporte |
   | `resolution_time` | Horas promedio para resolver tickets |
   | `active_months` | Meses activos desde el registro (>0) |
   | `churn` | 1 si canceló, 0 si sigue activo |

2. **compras.csv**
   | Columna | Descripción |
   |---|---|
   | `purchase_id` | ID de compra |
   | `user_id` | ID del usuario |
   | `purchase_amount` | Monto de la compra (USD) |
   | `purchase_date` | Fecha (YYYY-MM-DD) |

> **Importante:** `active_months` debe ser un entero > 0. Si encuentras 0 o nulos, ajusta en la limpieza para evitar divisiones por cero.

---

## 1) Carga, validación y exploración (10 pts)

**Objetivo:** Verificar estructura, tipos y consistencia entre tablas.
**Responde:**
1. ¿Cuántos **usuarios únicos** hay y cuántas **compras** totales?
2. ¿Cuántos **usuarios compraron al menos una vez**?
3. ¿Qué porcentaje representan sobre el total de usuarios?

In [ ]:
### TU CODIGO AQUI ###

import pandas as pd

# Carga
a_usuarios = pd.read_csv("usuarios.csv")
a_compras = pd.read_csv("compras.csv")

# Vistas rápidas
print(a_usuarios.shape); a_usuarios.head()
print(a_compras.shape); a_compras.head()

# Validaciones mínimas
assert a_usuarios['user_id'].is_unique, "user_id en usuarios debe ser único"
assert a_compras[['purchase_id']].drop_duplicates().shape[0] == a_compras.shape[0], "purchase_id en compras debe ser único"

usuarios = a_usuarios.copy()
compras = a_compras.copy()

# Métricas iniciales
n_usuarios = usuarios['user_id'].nunique()
n_compras = compras['purchase_id'].nunique()
usuarios_con_compra = compras['user_id'].nunique()

n_usuarios, n_compras, usuarios_con_compra

---

## 2) Limpieza de datos (15 pts)

**Objetivo:** Tratar nulos y outliers con criterios reproducibles.

* **Nulos a revisar:** `monthly_income`, `resolution_time`, `age`, `active_months`.
* **Outliers (IQR, factor 1.5):** `monthly_income`, `resolution_time`.

> **Fórmula IQR:**
> ( IQR = Q3 - Q1 )
> Límites: $[Q1 - 1.5\cdot IQR,\ Q3 + 1.5\cdot IQR]$


**Implementa (decisión justificada):**

* Limpia tu data de nulos, ya sea imputandolos o eliminandolos.
* Reemplaza `active_months` iguales a **0** por **1**.
* Realiza la limpieza de outliers.

In [ ]:
### TU CODIGO AQUI ###

import numpy as np

# Conteo de nulos
cols_nulos = ["monthly_income", "resolution_time", "age", "active_months"]
usuarios[cols_nulos].isna().sum()

# Función de límites IQR
def iqr_bounds(s: pd.Series):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5*iqr, q3 + 1.5*iqr
    return low, high

for col in ["monthly_income", "resolution_time"]:
    low, high = iqr_bounds(usuarios[col].dropna())
    outlier_rate = ((usuarios[col] < low) | (usuarios[col] > high)).mean()
    print(col, "outlier_rate=", round(float(outlier_rate), 4), "bounds=", (round(low, 2), round(high, 2)))

usuarios_clean = usuarios.copy()

# Nulos (ejemplo con mediana)
for col in ["monthly_income", "resolution_time", "age"]:
    if usuarios_clean[col].isna().mean() > 0:
        usuarios_clean[col] = usuarios_clean[col].fillna(usuarios_clean[col].median())

# active_months: prevenir 0
usuarios_clean['active_months'] = usuarios_clean['active_months'].fillna(1).clip(lower=1)

# Tratamiento de outliers por filtrado (ejemplo)
mask = pd.Series(True, index=usuarios_clean.index)
for col in ["monthly_income", "resolution_time"]:
    low, high = iqr_bounds(usuarios_clean[col])
    mask &= usuarios_clean[col].between(low, high)

usuarios_clean = usuarios_clean[mask].reset_index(drop=True)
usuarios.shape, usuarios_clean.shape

**Explica en 1-2 líneas tus decisiones de limpieza de datos.**

---

## 3) KPIs de rentabilidad y Churn (30 pts)

**Objetivo:** Construir KPIs a partir de la unión Usuarios–Compras.

1. **Une** usuarios y compras por `user_id`.
2. Calcula ingresos totales por usuario y **bandas mensuales**.
3. Calcula **Churn Rate** y **ARPU**

**Definiciones:**
- $\textbf{Churn Rate} = \frac{\sum churn}{\text{Número de usuarios}}$

- $\textbf{ARPU Mensual} = \frac{\text{Ingreso total mensual}}{\text{Número de usuarios}}$

**Visualizaciones obligatorias:**
1) **Churn Rate por `subscription_type`**
2) **ARPU Mensual por `signup_channel`**

In [ ]:
### TU CODIGO AQUI ###

# Union y agregaciones
compras["purchase_date"] = pd.to_datetime(compras["purchase_date"], errors="coerce")
ingresos_por_usuario = compras.groupby("user_id", as_index=False)["purchase_amount"].sum().rename(columns={"purchase_amount":"total_spent"})

base = usuarios_clean.merge(ingresos_por_usuario, on="user_id", how="left")
base["total_spent"] = base["total_spent"].fillna(0.0)

# KPIs
churn_rate = base['churn'].mean()
arpu = base['total_spent'].sum() / base['user_id'].nunique()
usuarios_pagadores = (base['total_spent'] > 0).sum()
arppu = base['total_spent'].sum() / max(usuarios_pagadores, 1)

churn_rate, arpu, arppu

import seaborn as sns
import matplotlib.pyplot as plt

fig = sns.barplot(data=base.groupby("subscription_type", as_index=False)["churn"].mean(), x="subscription_type", y="churn");
plt.title("Churn Rate por Tipo de Suscripción"); plt.xlabel(""); plt.ylabel("Churn Rate"); plt.show()

arpu_channel = base.groupby("signup_channel", as_index=False)["total_spent"].mean()
fig = sns.barplot(data=arpu_channel, x="signup_channel", y="total_spent");
plt.title("ARPU por Canal de Adquisición"); plt.xlabel(""); plt.ylabel("ARPU (USD)"); plt.show()

**Interpreta concisamente (3–5 líneas):** ¿Qué planes/canales son más rentables y cuáles retienen peor?

---

## 4) LTV por usuario y drivers (30 pts)

**Objetivo:** Calcular LTV a 6 meses  y explorar sus determinantes.

> **Definición usada en este examen:**
> $LTV_{\text{6 meses}} = \sum{\text{Total gastado los primeros 6 meses}}$

> *Recuerda que para el LTV a X meses solo se consideran los usuarios que ya tengan X periodo como clientes.*

**Tareas:**

1. Calcula `ltv` por usuario.
2. Identifica las **3 variables numéricas** de la tabla `usuarios` con mayor correlación con `ltv`.
3. Grafica **dispersión** de `ltv` vs. cada una de esas 3 variables.
4. Comenta patrones (3–5 líneas).

In [ ]:
### TU CODIGO AQUI ###

base["ltv_annual"] = (base["total_spent"] / base["active_months"]) * 12

corr = base.select_dtypes(include=["number"]).corr(numeric_only=True)["ltv_annual"].sort_values(ascending=False)
corr.head(10)

# Toma las 3 más correlacionadas excluyendo la diagonal (ltv_annual)
features = [c for c in corr.index if c != "ltv_annual"][:3]
features

for col in features:
    sns.scatterplot(data=base, x=col, y="ltv_annual")
    plt.title(f"LTV anual vs {col}")
    plt.xlabel(col); plt.ylabel("LTV anual (USD)")
    plt.show()

✏️ **Conclusión (3–5 líneas):** ¿Qué variables parecen explicar mejor el LTV? ¿Qué hipótesis de negocio sugieren?

---



## 5) Madurez analítica y recomendaciones ejecutivas (15 pts)

1. Propón **2 acciones** para **reducir churn** y **2 acciones** para **elevar LTV** (precisas, medibles).
2. Indica **qué dashboard/reportes** implementarías (métricas, frecuencia y audiencia).

TU RESPUESTA AQUI


---

## Criterios de Evaluación (100 pts)

* Carga/validación inicial (10)
* Limpieza de datos: nulos, outliers, justificación (15)
* KPIs: unión correcta, Churn, ARPU/ARPPU + 2 gráficas (30)
* LTV anualizado + correlaciones + 3 gráficos + interpretación (30)
* Madurez y memo ejecutivo con acciones (15)

> **Descuentos:** notebooks con errores de ejecución, sin gráficos o sin justificación restan puntos.

---